# Week 3 — 01: Data Preparation

Load raw HI-Small transactions, parse the laundering-pattern file, build the
pattern-to-row map, and apply the leakage-safe chronological split.

**Outputs saved to `checkpoints/`:**
- `01_df.pkl`  — full transaction DataFrame with `split` column
- `01_pattern_map.pkl` — pattern-to-row mapping
- `01_indices.pkl`  — (train_idx, val_idx, test_idx)


In [1]:
import sys, pathlib

# Resolve project root so notebooks work regardless of CWD or kernel launch dir
_this_nb  = pathlib.Path(__file__) if "__file__" in dir() else pathlib.Path().resolve()
_sentinel = "src"   # directory that marks the project root
_root     = _this_nb.resolve()
for _ in range(10):                          # walk up at most 10 levels
    if (_root / _sentinel).is_dir():
        break
    _root = _root.parent

# Hard-code fallback for direct kernel launch from notebooks/week3/
_fallback = pathlib.Path("/home/malnaser/Eastnets-Standard-Workspace/DEVELOPMENT/_playground/DS-Internship-Summer-2026/aml-detection-project")
if not (_root / _sentinel).is_dir():
    _root = _fallback

if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

PROJECT_ROOT = _root
print("Project root:", PROJECT_ROOT)


Project root: /home/malnaser/EN_Workspace/DEVELOPMENT/_playground/DS-Internship-Summer-2026/aml-detection-project


In [2]:
import pickle, numpy as np
from tqdm.auto import tqdm
from src.data_loading import load_transactions, parse_pattern_file, build_pattern_map
from src.splitting import build_split
from src import state

# ── Load transactions ─────────────────────────────────────────────────────────
df = load_transactions()
print(f"Rows: {len(df):,}  |  Positives: {int(df.target.sum()):,}  |  Rate: {df.target.mean():.6%}")
print(f"Date range: {df.ts.min()} -> {df.ts.max()}")


Reading HI-Small CSV: 0chunk [00:00, ?chunk/s]

Rows: 5,078,345  |  Positives: 5,177  |  Rate: 0.101943%
Date range: 2022-09-01 00:00:00 -> 2022-09-18 16:18:00


In [3]:
# ── Parse patterns ────────────────────────────────────────────────────────────
attempts = parse_pattern_file()
print(f"Pattern attempts: {len(attempts)}")

import pandas as pd
print(pd.Series([a["pattern_type"] for a in attempts]).value_counts())


Parsing pattern file: 0line [00:00, ?line/s]

Pattern attempts: 370
BIPARTITE                               49
SCATTER-GATHER                          44
STACK                                   43
CYCLE:  MAX 2 HOPS                      14
RANDOM:  MAX 1 HOPS                     13
                                        ..
FAN-OUT:  MAX 6-DEGREE FAN-OUT           1
FAN-OUT:  MAX 14-DEGREE FAN-OUT          1
GATHER-SCATTER:  MAX 9-DEGREE FAN-IN     1
FAN-OUT:  MAX 15-DEGREE FAN-OUT          1
GATHER-SCATTER:  MAX 5-DEGREE FAN-IN     1
Name: count, Length: 65, dtype: int64


In [4]:
# ── Build pattern map (cached) ────────────────────────────────────────────────
if state.exists("01_pattern_map"):
    pattern_map = state.load("01_pattern_map")
else:
    pattern_map = build_pattern_map(df, attempts)
    state.save("01_pattern_map", pattern_map)


Loaded checkpoint: /home/malnaser/EN_Workspace/DEVELOPMENT/_playground/DS-Internship-Summer-2026/aml-detection-project/checkpoints/01_pattern_map.pkl


In [5]:
# ── Build leakage-safe split ─────────────────────────────────────────────────
df, pattern_map = build_split(df, pattern_map)

y         = df.target.to_numpy(dtype=np.int8)
train_idx = np.flatnonzero(df["split"].eq("train").to_numpy())
val_idx   = np.flatnonzero(df["split"].eq("val").to_numpy())
test_idx  = np.flatnonzero(df["split"].eq("test").to_numpy())

print(df.groupby("split")["target"].agg(["count","sum","mean"]))
print("Pattern components:", pattern_map.pattern_component.nunique())

state.save("01_df",          df)
state.save("01_pattern_map", pattern_map)
state.save("01_indices",     (train_idx, val_idx, test_idx))
print("Checkpoint 01 saved.")


Building pattern components:   0%|          | 0/3209 [00:00<?, ?transaction/s]

Assigning pattern components to splits:   0%|          | 0/370 [00:00<?, ?component/s]

         count   sum      mean
split                         
test   1015834  1265  0.001245
train  3046810  2849  0.000935
val    1015701  1063  0.001047
Pattern components: 370
Saved checkpoint: /home/malnaser/EN_Workspace/DEVELOPMENT/_playground/DS-Internship-Summer-2026/aml-detection-project/checkpoints/01_df.pkl
Saved checkpoint: /home/malnaser/EN_Workspace/DEVELOPMENT/_playground/DS-Internship-Summer-2026/aml-detection-project/checkpoints/01_pattern_map.pkl
Saved checkpoint: /home/malnaser/EN_Workspace/DEVELOPMENT/_playground/DS-Internship-Summer-2026/aml-detection-project/checkpoints/01_indices.pkl
Checkpoint 01 saved.
